# SceneFlow Dataset Downloader & Reformatter for Google Colab
This notebook automates downloading the SceneFlow datasets (FlyingThings3D and Driving) using BitTorrent (`aria2c`) and extracting them directly to your mounted Google Drive in the format expected by the `bridgedepth` data loader.

## Sequential Processing to Avoid Disk Exhaustion:
Because Colab has limited local disk space, downloading all archives at once is not possible. This notebook processes each archive **sequentially**:
1. Downloads the compressed archive (`.tar` or `.tar.bz2`) using `aria2c` via torrent or direct link to local Colab storage `/content/temp_downloads/`.
2. Detects the archive's internal directory layout.
3. Extracts it directly to your mounted Google Drive using the highly optimized C-based `tar` tool.
4. Deletes the local compressed archive before proceeding to the next download.
This ensures that local disk usage never exceeds the size of a single compressed archive (~30 GB).

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Install aria2 downloader (very fast, supports HTTP, FTP, and BitTorrent)
!apt-get update && apt-get install -y aria2

In [ ]:
import os
import urllib.parse
import tarfile
from pathlib import Path

# ── Target Directory inside Google Drive ──────────────────────────────────
# The bridgedepth data loader expects: 
#  - FlyingThings3D images inside: sceneflow_root/FlyingThings3D/frames_cleanpass/
#  - Driving images inside: sceneflow_root/Driving/frames_cleanpass/
GDRIVE_SCENEFLOW_ROOT = "/content/drive/MyDrive/sceneflow"

# Temp directory on local Colab disk to store compressed downloads
LOCAL_TEMP_DIR = "/content/temp_downloads"

# List of download jobs. Set enable=False if you want to skip any specific parts.
DOWNLOAD_JOBS = [
    {
        "name": "FlyingThings3D Disparity",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/derived_data/flyingthings3d__disparity.tar.bz2.torrent",
        "filename": "flyingthings3d__disparity.tar.bz2",
        "is_torrent": True,
        "expected_dir": "FlyingThings3D",
        "enabled": True
    },
    {
        "name": "FlyingThings3D RGB Cleanpass Images",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/raw_data/flyingthings3d__frames_cleanpass.tar.torrent",
        "filename": "flyingthings3d__frames_cleanpass.tar",
        "is_torrent": True,
        "expected_dir": "FlyingThings3D",
        "enabled": True
    },
    {
        "name": "FlyingThings3D RGB Finalpass Images",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/raw_data/flyingthings3d__frames_finalpass.tar.torrent",
        "filename": "flyingthings3d__frames_finalpass.tar",
        "is_torrent": True,
        "expected_dir": "FlyingThings3D",
        "enabled": True
    },
    {
        "name": "FlyingThings3D Camera Data",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/CameraData_august16/data/FlyingThings3D/raw_data/flyingthings3d__camera_data.tar",
        "filename": "flyingthings3d__camera_data.tar",
        "is_torrent": False,
        "expected_dir": "FlyingThings3D",
        "enabled": True
    },
    {
        "name": "Driving RGB Cleanpass Images",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Driving/raw_data/driving__frames_cleanpass.tar.torrent",
        "filename": "driving__frames_cleanpass.tar",
        "is_torrent": True,
        "expected_dir": "Driving",
        "enabled": True
    },
    {
        "name": "Driving RGB Finalpass Images",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Driving/raw_data/driving__frames_finalpass.tar.torrent",
        "filename": "driving__frames_finalpass.tar",
        "is_torrent": True,
        "expected_dir": "Driving",
        "enabled": True
    },
    {
        "name": "Driving Disparity",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Driving/derived_data/driving__disparity.tar.bz2.torrent",
        "filename": "driving__disparity.tar.bz2",
        "is_torrent": True,
        "expected_dir": "Driving",
        "enabled": True
    },
    {
        "name": "Driving Camera Data",
        "url": "https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/CameraData_august16/data/Driving/raw_data/driving__camera_data.tar",
        "filename": "driving__camera_data.tar",
        "is_torrent": False,
        "expected_dir": "Driving",
        "enabled": True
    }
]

In [ ]:
def download_file(url: str, filename: str, is_torrent: bool, temp_dir: str):
    os.makedirs(temp_dir, exist_ok=True)
    local_path = os.path.join(temp_dir, filename)
    
    # Clean up existing files if they exist
    if os.path.exists(local_path):
        print(f"[downloader] File already exists locally, removing: {local_path}")
        os.remove(local_path)
        
    if is_torrent:
        torrent_path = os.path.join(temp_dir, "temp.torrent")
        if os.path.exists(torrent_path):
            os.remove(torrent_path)
            
        print(f"[downloader] Fetching torrent file meta: {url}")
        # Download torrent file
        import requests
        r = requests.get(url)
        with open(torrent_path, "wb") as f:
            f.write(r.content)
            
        # Run BitTorrent via aria2c
        print(f"[downloader] Initiating BitTorrent download for: {filename}")
        # --seed-time=0 stops uploading/seeding immediately after download completes
        cmd = f"aria2c --seed-time=0 -d {temp_dir} {torrent_path}"
        status = os.system(cmd)
        if status != 0:
            raise RuntimeError(f"BitTorrent download failed for {filename}")
    else:
        print(f"[downloader] Initiating direct HTTP download for: {filename}")
        cmd = f"aria2c -d {temp_dir} -o {filename} {url}"
        status = os.system(cmd)
        if status != 0:
            raise RuntimeError(f"Direct HTTP download failed for {filename}")
            
    return local_path


def extract_and_format_archive(archive_path: str, dest_root: str, expected_dir: str):
    print(f"[extractor] Inspecting archive: {archive_path}")
    
    # Check top-level folder inside tar archive
    with tarfile.open(archive_path, 'r') as tar:
        first_member = tar.next()
        if first_member:
            top_dir = first_member.name.split('/')[0]
        else:
            top_dir = ""
            
    # Decide extraction folder layout to match dataloader structure
    # (If the tar contains 'FlyingThings3D' as its root directory, extract to dest_root.)
    # (Otherwise, extract inside dest_root/FlyingThings3D.)
    if top_dir.lower() == expected_dir.lower():
        extract_path = dest_root
    else:
        extract_path = os.path.join(dest_root, expected_dir)
        
    os.makedirs(extract_path, exist_ok=True)
    print(f"[extractor] Extracting directory layout to Google Drive destination: {extract_path}")
    
    # Use highly optimized Unix tar CLI command
    if archive_path.endswith('.bz2'):
        cmd = f"tar -xjf {archive_path} -C {extract_path}"
    else:
        cmd = f"tar -xf {archive_path} -C {extract_path}"
        
    print(f"[extractor] Executing extraction command: {cmd}")
    status = os.system(cmd)
    if status != 0:
        raise RuntimeError(f"Extraction failed for {archive_path}")
    print(f"[extractor] Successfully extracted and formatted layout!")

In [ ]:
os.makedirs(GDRIVE_SCENEFLOW_ROOT, exist_ok=True)

for idx, job in enumerate(DOWNLOAD_JOBS):
    if not job["enabled"]:
        print(f"\n>>> Skipping job {idx+1}/{len(DOWNLOAD_JOBS)}: {job['name']}")
        continue
        
    print("\n" + "=" * 80)
    print(f"  JOB {idx+1}/{len(DOWNLOAD_JOBS)}: {job['name']}")
    print("=" * 80)
    
    try:
        # 1. Download local file
        local_archive = download_file(
            url=job["url"],
            filename=job["filename"],
            is_torrent=job["is_torrent"],
            temp_dir=LOCAL_TEMP_DIR
        )
        
        # 2. Extract and format directly to Google Drive
        extract_and_format_archive(
            archive_path=local_archive,
            dest_root=GDRIVE_SCENEFLOW_ROOT,
            expected_dir=job["expected_dir"]
        )
        
        # 3. Delete local compressed file to free disk space
        print(f"[cleanup] Deleting local archive to save disk space: {local_archive}")
        if os.path.exists(local_archive):
            os.remove(local_archive)
            
    except Exception as e:
        print(f"[ERROR] Job {job['name']} failed: {e}")
        print("Stopping pipeline to prevent data mismatch. Please resolve and resume.")
        break
        
print("\n" + "=" * 80)
print("  PIPELINE PROCESSING COMPLETED")
print("=" * 80)
